In [1]:
from ultralytics import YOLO

# 1. 加载你训练好的权重 (Standard DP)
model_dp = YOLO('/root/autodl-tmp/exp1/result_exp1/18.2_Transfer_BasicDP_Ablation/weights/best.pt') # 替换成你的权重路径
# 2. 在验证集/测试集上运行验证
metrics_dp = model_dp.val(data='/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml', split='test', plots=True)
# 运行完后，去 runs/detect/val/ 文件夹下找 confusion_matrix.png

# 3. 加载你的方法权重 (Ours)
model_ours = YOLO('/root/autodl-tmp/exp1/result_exp1/18_Rescue_Crazing/weights/best.pt')
# 4. 运行验证
metrics_ours = model_ours.val(data='/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml', split='test', plots=True)

Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 102.6±23.5 MB/s, size: 15.6 KB)
val: Scanning /root/autodl-tmp/exp1/NEU_DET_YOLO/labels/test... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 1.5Kit/s 0.1s<0.1s
val: New cache created: /root/autodl-tmp/exp1/NEU_DET_YOLO/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 6.1it/s 2.0s<0.1s
                   all        180        436      0.504      0.328      0.289       0.11
               crazing         30         74      0.778      0.027      0.176       0.07
             inclusion         42        110      0.219      0.291      0.243      0.101
               patches         28         81      0.416      0.469        0.4      0.189
        pitted_surface    

In [2]:
from ultralytics import YOLO
import numpy as np

def get_confusion_matrix_data(weights_path, data_yaml):
    # 加载模型
    model = YOLO(weights_path)
    
    # 运行验证 (设置 plots=False 以节省时间，我们只要数据)
    # save_json=True 有助于后续分析，但这里我们直接读对象
    metrics = model.val(data=data_yaml, split='test', plots=False)
    
    # 获取混淆矩阵对象
    cm_object = metrics.confusion_matrix
    
    # 获取原始矩阵数组 (行是真实标签，列是预测标签)
    # 形状通常是 (N+1) x (N+1)，最后一行/列是背景
    matrix = cm_object.matrix 
    
    # 获取类别名称
    names = list(model.names.values())
    
    return matrix, names

# ================= 配置路径 =================
# 请替换为你的实际路径
weights_dp = '/root/autodl-tmp/exp1/result_exp1/18.2_Transfer_BasicDP_Ablation/weights/best.pt' 
weights_ours = '/root/autodl-tmp/exp1/result_exp1/18_Rescue_Crazing/weights/best.pt'
data_yaml = '/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml' 
# ===========================================

print("正在处理 Standard DP 模型...")
matrix_dp, names = get_confusion_matrix_data(weights_dp, data_yaml)

print("\n正在处理 ST-ADM (Ours) 模型...")
matrix_ours, _ = get_confusion_matrix_data(weights_ours, data_yaml)

print("\n====== Standard DP 原始矩阵数据 ======")
print(matrix_dp)
print("\n====== ST-ADM 原始矩阵数据 ======")
print(matrix_ours)
print("\n====== 类别顺序 ======")
print(names)

正在处理 Standard DP 模型...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 633.6±252.6 MB/s, size: 16.8 KB)
val: Scanning /root/autodl-tmp/exp1/NEU_DET_YOLO/labels/test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 279.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 11.8it/s 1.0s0.1s
                   all        180        436      0.504      0.328      0.289       0.11
               crazing         30         74      0.778      0.027      0.176       0.07
             inclusion         42        110      0.219      0.291      0.243      0.101
               patches         28         81      0.416      0.469        0.4      0.189
        pitted_surface         24         32      0.231      0.906      0.54

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ================= 配置路径 =================
# 把这里换成你真实的 csv 路径
# 注意：YOLO v8/v11 的 csv 列名可能包含空格，pandas读取时要注意
files = {
    'Standard DP (Baseline)': '/root/autodl-tmp/exp1/result_exp1/18.2_Transfer_BasicDP_Ablation/results.csv', # 替换路径
    'ST-ADM (Ours)':          '/root/autodl-tmp/exp1/result_exp1/18_Rescue_Crazing/results.csv'       # 替换路径
}
# ===========================================

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid") # 设置学术风格背景

# 定义颜色
colors = {'Standard DP (Baseline)': '#FF6B6B', 'ST-ADM (Ours)': '#4ECDC4'}

for label, filepath in files.items():
    try:
        # 读取 CSV
        df = pd.read_csv(filepath)
        
        # YOLO 的 csv 列名通常带有空格，比如 "   metrics/mAP50(B)"
        # 我们需要清洗一下列名，去掉空格
        df.columns = [c.strip() for c in df.columns]
        
        # 找到 epoch 和 mAP50 列
        # YOLO v8/v11 通常叫 'metrics/mAP50(B)'
        # YOLO v5 通常叫 'metrics/mAP_0.5'
        # 这里做一个简单的自动查找
        map_col = [c for c in df.columns if 'mAP50' in c and '95' not in c][0]
        
        # 绘制曲线
        sns.lineplot(x=df['epoch'], y=df[map_col], label=label, color=colors[label], linewidth=2.5)
        
    except Exception as e:
        print(f"读取 {label} 失败: {e}")
        print(f"请检查路径: {filepath}")

# 美化图表
plt.title("Training Convergence Analysis (mAP@0.5)", fontsize=16, fontweight='bold')
plt.xlabel("Epochs", fontsize=14)
plt.ylabel("mAP@0.5", fontsize=14)
plt.legend(fontsize=12, loc='lower right')
plt.xlim(0, None)
plt.ylim(0, 1.0) # 或者是你的最大精度

# 加上网格
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

<Figure size 1000x600 with 1 Axes>